# SciAR dataset builder

Build a frozen, refereed-only paper dataset for **2014–2024 inclusive**, with at least **two distinct papers per first-author ORCID**. Preserve the original text, exclude conflicting candidate IDs, and deduplicate by bibcode. No train/test split or pairs are created yet.

ORCIDs are checked for format only. First-author identity is a label proxy, not evidence of who wrote the text. Source ORCID lists are assumed to follow author order when their lengths match the author list.


In [22]:
from pathlib import Path
from datetime import datetime, timezone
import json
import duckdb
from IPython.display import display

# Configuration

In [23]:
DATA_DIR = Path("/home/petrecc1/projects/SciAR/NASA_ADS_UNIFIED")
OUTPUT_DIR = Path(
    "/home/petrecc1/projects/SciAR/data/processed/sciar_refereed_v0"
)

YEARS = list(range(2014, 2025))
MIN_PAPERS_PER_AUTHOR = 1
EXCLUDED_ORCIDS = ["0000-0001-2345-6789"]

THREADS = 64
MEMORY_LIMIT = "256GB"
TEMP_DIR = Path("/home/petrecc1/projects/SciAR/.duckdb_tmp")

assert DATA_DIR.is_dir(), f"Source directory not found: {DATA_DIR}"
assert YEARS and MIN_PAPERS_PER_AUTHOR >= 1

if OUTPUT_DIR.exists():
    raise FileExistsError(
        f"{OUTPUT_DIR} already exists. Choose a new output version."
    )

TEMP_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"SET threads = {THREADS}")
con.execute("SET memory_limit = ?", [MEMORY_LIMIT])
con.execute("SET temp_directory = ?", [str(TEMP_DIR)])

# Build the dataset

In [24]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE papers AS
    WITH candidates AS (
        SELECT
            bibcode,
            year AS partition_year,
            first_author,
            list_contains(property, 'REFEREED') AS is_refereed,
            list_filter(
                title, lambda t: length(trim(t)) > 0
            )[1] AS title,
            abstract,
            list_distinct(list_filter([
                CASE WHEN len(orcid_pub) = len(author)
                     THEN trim(orcid_pub[1]) END,
                CASE WHEN len(orcid_user) = len(author)
                     THEN trim(orcid_user[1]) END,
                CASE WHEN len(orcid_other) = len(author)
                     THEN trim(orcid_other[1]) END
            ], lambda x: regexp_full_match(
                x, '[0-9]{4}-[0-9]{4}-[0-9]{4}-[0-9]{3}[0-9X]'
            ))) AS orcids
        FROM read_parquet(
            ?, hive_partitioning=true, union_by_name=true
        )
        WHERE year IN (SELECT unnest(?))
          AND list_contains(property, 'REFEREED')
          AND first_author = author[1]
          AND length(list_filter(
              title, lambda t: length(trim(t)) > 0
          )) > 0
          AND length(trim(abstract)) > 0
          AND bibcode IS NOT NULL
          AND trim(bibcode) <> ''
    ),
    checked AS (
        SELECT *,
            list_distinct(
                flatten(list(orcids) OVER (PARTITION BY bibcode))
            ) AS all_candidate_ids
        FROM candidates
    ),
    deduplicated AS (
        SELECT
            bibcode,
            partition_year,
            first_author,
            orcids[1] AS first_author_orcid,
            title,
            abstract,
            is_refereed
        FROM checked
        WHERE len(orcids) = 1
          AND len(all_candidate_ids) = 1
          AND orcids[1] NOT IN (SELECT unnest(?))
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY bibcode
            ORDER BY partition_year, title, abstract, first_author
        ) = 1
    )
    SELECT *
    FROM deduplicated
    QUALIFY COUNT(*) OVER (
        PARTITION BY first_author_orcid
    ) >= ?
""", [
    str(DATA_DIR),
    YEARS,
    EXCLUDED_ORCIDS,
    MIN_PAPERS_PER_AUTHOR,
])

# Validate

In [25]:
paper_count, author_count, unique_bibcodes, all_refereed = con.execute("""
    SELECT
        COUNT(*),
        COUNT(DISTINCT first_author_orcid),
        COUNT(DISTINCT bibcode),
        bool_and(is_refereed)
    FROM papers
""").fetchone()

assert paper_count > 0, "Dataset is empty."
assert paper_count == unique_bibcodes, "Duplicate bibcodes found."
assert all_refereed, "Nonrefereed record found."

assert con.execute("""
    SELECT MIN(n) >= ?
    FROM (
        SELECT COUNT(*) AS n
        FROM papers
        GROUP BY first_author_orcid
    )
""", [MIN_PAPERS_PER_AUTHOR]).fetchone()[0], (
    "Author below minimum paper count."
)

assert con.execute("""
    SELECT COUNT(*) = 0
    FROM papers
    WHERE first_author_orcid IS NULL
       OR first_author_orcid IN (SELECT unnest(?))
       OR title IS NULL OR length(trim(title)) = 0
       OR abstract IS NULL OR length(trim(abstract)) = 0
       OR partition_year NOT IN (SELECT unnest(?))
""", [EXCLUDED_ORCIDS, YEARS]).fetchone()[0], "Invalid record found."
con.complete()
display(con.sql("""
    SELECT
        COUNT(*) AS papers,
        COUNT(DISTINCT first_author_orcid) AS authors,
        MIN(partition_year) AS earliest_partition,
        MAX(partition_year) AS latest_partition
    FROM papers
""").df())

,papers,authors,earliest_partition,latest_partition
0,1175773,718924,2014,2024


# Export without overwriting a previous version

In [26]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

parquet_path = OUTPUT_DIR / "papers.parquet"
sql_path = str(parquet_path).replace("'", "''")

con.execute(f"""
    COPY (
        SELECT * FROM papers
        ORDER BY first_author_orcid, bibcode
    )
    TO '{sql_path}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

saved_count = con.execute("""
    SELECT COUNT(*) FROM read_parquet(?)
""", [str(parquet_path)]).fetchone()[0]

assert saved_count == paper_count, "Export row count mismatch."

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source": str(DATA_DIR),
    "years": YEARS,
    "year_definition": "Hive directory partition",
    "minimum_papers_per_author": MIN_PAPERS_PER_AUTHOR,
    "excluded_first_author_orcids": EXCLUDED_ORCIDS,
    "papers": paper_count,
    "authors": author_count,
    "identity": "First-author ORCID; format checked, not checksum or externally verified",
    "conflicts": "Conflicting candidate IDs excluded within and across eligible duplicate records",
    "deduplication": "One record per bibcode; deterministic text selection",
    "text": "First nonblank title and original abstract; no text cleaning",
    "refereed_only": True,
    "doctype_filter": None,
    "split": "Not yet assigned",
    "duckdb_version": duckdb.__version__,
}

(OUTPUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print(f"Saved {paper_count:,} refereed papers across {author_count:,} authors")
print(parquet_path)

Saved 1,175,773 refereed papers across 718,924 authors
/home/petrecc1/projects/SciAR/data/processed/sciar_refereed_v0/papers.parquet
